# Interactive Multi-Agent Workflow - Sales Assist Tool

**Workflow:** Seller Query → Supervisory Agent → Contract Agent → Research Agent → Matching Agent → Action Agent → Results

## How to Use This Notebook

1. **Run Setup** - Install packages and initialize agents
2. **Step 1** - Ask your initial query
3. **Step 2** - Review and edit the draft email
4. **Step 3** - Confirm and send the email

---
## Setup and Environment Configuration

**Note**: The Contract Agent will automatically use cached data from `contracts_cache.json` if available!

---
## Step 0: Generate Contract Cache (First Time Only)

**IMPORTANT**: Run this cell ONCE to extract and cache all contract data.

**What it does**: Extracts text and structured fields (amount, products, dates) from contracts and saves to `contracts_cache.json`.

**Time**: ~1 minute (no LLM calls!)

**When to re-run**: Only when contract files change in `docs/` folder.

In [1]:
import os
import subprocess
import sys

cache_exists = os.path.exists("contracts_cache.json")

if cache_exists:
    print("="*80)
    print("CONTRACT CACHE ALREADY EXISTS")
    print("="*80)
    print("Cache file found: contracts_cache.json")
    print("✓ Contracts will load instantly from cache")
    print("To regenerate cache (if contracts changed):")
    print("  1. Delete contracts_cache.json")
    print("  2. Run: python cache_contracts.py")
    print("="*80)
else:
    print("="*80)
    print("GENERATING CONTRACT CACHE")
    print("="*80)
    print("Extracting text and structured fields from contracts...")
    print("This only needs to be done ONCE.")
    
    try:
        result = subprocess.run(
            [sys.executable, "cache_contracts.py"],
            capture_output=True,
            text=True,
            timeout=300
        )
        
        if result.returncode == 0:
            print("✓ Cache generated successfully!")
            print("✓ Future runs will be 5-10x faster")
        else:
            print(f"Warning: {result.stderr}")
            print("Run manually: python cache_contracts.py")
    except Exception as e:
        print(f"Could not auto-generate: {e}")
        print("Run manually: python cache_contracts.py")
    
    print("="*80)

CONTRACT CACHE ALREADY EXISTS
Cache file found: contracts_cache.json
✓ Contracts will load instantly from cache
To regenerate cache (if contracts changed):
  1. Delete contracts_cache.json
  2. Run: python cache_contracts.py


In [2]:
# Install all requirements from requirements.txt
import sys
import subprocess

print("Installing packages from requirements.txt...")
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    print("All packages installed successfully!\n")
except subprocess.CalledProcessError as e:
    print(f"Error installing packages: {e}\n")
except FileNotFoundError:
    print("requirements.txt file not found!\n")

# Import required libraries
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

# Verify credentials
print("Environment Check:")
print(f"WATSONX_APIKEY: {'Set' if os.getenv('WATSONX_APIKEY') else 'Missing'}")
print(f"WATSONX_PROJECT_ID: {'Set' if os.getenv('WATSONX_PROJECT_ID') else 'Missing'}")
print(f"TAVILY_API_KEY: {'Set' if os.getenv('TAVILY_API_KEY') else 'Missing'}")

Installing packages from requirements.txt...
All packages installed successfully!

Environment Check:
WATSONX_APIKEY: Set
WATSONX_PROJECT_ID: Set
TAVILY_API_KEY: Set


## Initialize the Supervisory Agent

The Supervisory Agent orchestrates all other agents in the workflow.

In [3]:
from supervisory_agent import SupervisoryAgent
import langchain_chroma

# Initialize the Supervisory Agent
print("Initializing Supervisory Agent...")
supervisor = SupervisoryAgent(
    apikey=os.getenv("WATSONX_APIKEY"),
    project_id=os.getenv("WATSONX_PROJECT_ID")
)
print("✓ Supervisory Agent ready\n")

Initializing Supervisory Agent...
✓ Supervisory Agent ready



## Initialize watsonx.governance Evaluator

Set up the governance evaluator for quality assessment of agent outputs.

In [4]:
from ibm_watsonx_gov.evaluators.metrics_evaluator import MetricsEvaluator
from ibm_watsonx_gov.metrics import FaithfulnessMetric
from ibm_watsonx_gov.config import GenAIConfiguration
from ibm_watsonx_gov.entities.foundation_model import WxAIFoundationModel
from ibm_watsonx_gov.entities.llm_judge import LLMJudge

PROJECT_ID = os.getenv("WATSONX_PROJECT_ID")
REGION = "us-south"  

# Use Llama 3.3 70B for LLM judge (more stable than Llama 4 FP8 for evaluation)
# Keep Llama 4 for agents, but use Llama 3.3 for judging
llm_judge = LLMJudge(
    model=WxAIFoundationModel(
        model_id="meta-llama/llama-3-3-70b-instruct",
        project_id=PROJECT_ID,
        region=REGION
    )
)

evaluator = MetricsEvaluator(
    project_id=PROJECT_ID,
    region=REGION
)


[2026-04-22 15:30:38,234]-[ibm_watsonx_gov.evaluators.agentic_evaluator]-[ WARNING ]-[Line 125] ~~> No module named 'ibm_agent_analytics'


---
## Interactive Workflow

### Step 1: Ask Your Initial Query

Enter your query below and run the cell to execute the full multi-agent workflow.

In [5]:
my_query = "I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps"

print("="*80)
print("YOUR QUERY")
print("="*80)
print(f"\n{my_query}\n")
print("="*80)
print("Executing full workflow...")
print("="*80)

my_result = supervisor.run(
    seller_query=my_query,
    contract_file_path=None,
    partner_name="Confluent"
)

print("\n" + "="*80)
print("WORKFLOW EXECUTION COMPLETE")
print("="*80 + "\n")
print("✓ All agents executed successfully")
print("✓ Results available for detailed review in cells below")
print("\nScroll down to see:")
print("  • Contract Portfolio Summary")
print("  • Partner Profile & CRM Data")
print("  • Contract-CRM Matching Results")
print("  • Action Recommendations & Reasoning")
print("  • Draft Email")
print("  • CRM Updates")

YOUR QUERY

I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps

Executing full workflow...

SUPERVISORY AGENT - Workflow Initialization
Seller Query: I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps

Workflow Type: renewal_expiration_awareness
Required Agents: contract, action
Partner Name: Confluent

EXECUTING CONTRACT AGENT
Preloading contract portfolio for partner: Confluent
Contract scope: all files in docs/ beginning with Confluent_IBM
✓ Using cached data for: Confluent_IBM-1.30.2024.docx
✓ Using cached metadata for: Confluent_IBM-

### Result Component 1: Contract Portfolio Summary

View all contracts analyzed by the Contract Agent.

In [6]:
import json

contract_summary = my_result.get('contract_summary', {})
portfolio_summary = contract_summary.get('portfolio_summary', {})

print("="*80)
print("CONTRACT PORTFOLIO SUMMARY")
print("="*80)

if portfolio_summary:
    print(f"\nTotal Contracts: {portfolio_summary.get('total_contracts', 0)}")
    print(f"Active Contracts: {len(portfolio_summary.get('active_contracts', []))}")
    print(f"Renewal Candidates: {len(portfolio_summary.get('renewal_candidates', []))}")
    print(f"Recently Expired: {len(portfolio_summary.get('recently_expired_contracts', []))}")
    
    all_contracts = portfolio_summary.get('contract_results', [])
    if all_contracts:
        print("\n" + "="*80)
        print("CONTRACT DETAILS")
        print("="*80)
        for i, contract in enumerate(all_contracts, 1):
            structured = contract.get('structured_summary', {})
            print(f"\n{i}. {contract.get('file_name', 'Unknown')}")
            print(f"   Product(s): {', '.join(structured.get('products', ['Unknown']))}")
            print(f"   Amount: {structured.get('amount', 'Not specified')}")
            print(f"   Start Date: {contract.get('effective_date', 'Unknown')}")
            print(f"   End Date: {contract.get('end_date', 'Unknown')}")
            print(f"   Status: {contract.get('status', 'Unknown').upper()}")
            print(f"   Days to End: {contract.get('days_to_end', 'N/A')}")
else:
    print("\nNo contract data available")

CONTRACT PORTFOLIO SUMMARY

Total Contracts: 4
Active Contracts: 2
Renewal Candidates: 2
Recently Expired: 2


### Result Component 2: Partner Profile & CRM Data

View partner intelligence and CRM opportunities from the Research Agent.

In [7]:
partner_profile = my_result.get('partner_profile', {})
internal_data = partner_profile.get('internal_data', {})
sales_history = internal_data.get('sales_history', {})
opportunities = sales_history.get('opportunities', [])

print("="*80)
print("PARTNER PROFILE & CRM DATA")
print("="*80)

if partner_profile:
    print(f"\nPartner: {partner_profile.get('partner_name', 'Unknown')}")
    print(f"Maturity Level: {partner_profile.get('maturity_level', 'Unknown')}")
    print(f"Sales Velocity: {partner_profile.get('sales_velocity', 'Unknown')}")
    
    if opportunities:
        print(f"\n{'='*80}")
        print("CRM OPPORTUNITIES")
        print("="*80)
        print(f"Total Opportunities: {len(opportunities)}\n")
        
        for i, opp in enumerate(opportunities, 1):
            amount = opp.get('amount', 0)
            if isinstance(amount, (int, float)):
                amount_str = f"${amount:,.0f}"
            else:
                amount_str = str(amount) if amount else "$0"
            
            opp_num = opp.get('opportunity_number', i)
            print(f"{i}. CRM #{opp_num}: {opp.get('opportunity_name', 'Unknown')}")
            print(f"   Owner: {opp.get('owner', 'Unknown')}")
            print(f"   Stage: {opp.get('stage', 'Unknown')}")
            print(f"   Amount: {amount_str}")
            print(f"   Close Date: {opp.get('close_date', 'Unknown')}")
            print(f"   Products: {opp.get('products', 'Unknown')}")
            print(f"   Next Steps: {opp.get('next_steps', 'None specified')}")
            print()
else:
    print("\nNo partner profile data available")

PARTNER PROFILE & CRM DATA

Partner: Confluent
Maturity Level: Engaged Prospect
Sales Velocity: High

CRM OPPORTUNITIES
Total Opportunities: 12

1. CRM #1: Confluent Cognos
   Owner: Kylie Brittz
   Stage: Won
   Amount:  $1,000,000.00 
   Close Date: 5/31/23
   Products: Cognos
   Next Steps: Deal signed by CPO

2. CRM #2: Confluent watsonx ESA
   Owner: Anand Das
   Stage: Won
   Amount:  $250,000.00 
   Close Date: 1/31/25
   Products: watsonx Orchestrate, watsonx.governance, watsonx.ai
   Next Steps: CFO signed 1 year renewals

3. CRM #3: Confluent watsonx ESA
   Owner: Anand Das
   Stage: Won
   Amount:  $250,000.00 
   Close Date: 1/31/24
   Products: watsonx Orchestrate, watsonx.governance, watsonx.ai
   Next Steps: CFO signed 1 year deal for them embedding watsonx to add agentic and AI use cases to their solution

4. CRM #4: Confluent watsonx ESA Expansion
   Owner: Anand Das
   Stage: Won
   Amount:  $400,000.00 
   Close Date: 7/31/24
   Products: watsonx.data
   Next Steps: 

### Result Component 3: Contract-CRM Matching Results

View how contracts correlate with CRM opportunities from the Matching Agent.

In [8]:
matching_data = my_result.get('matching_data', {})

print("="*80)
print("CONTRACT-CRM MATCHING RESULTS")
print("="*80)

if matching_data and not matching_data.get('error'):
    matched = matching_data.get('matched_contracts', [])
    unmatched = matching_data.get('unmatched_contracts', [])
    
    print(f"\nMatched Contracts: {len(matched)}")
    print(f"Unmatched Contracts: {len(unmatched)}")
    
    if matched:
        print(f"\n{'='*80}")
        print("MATCHED CONTRACTS")
        print("="*80)
        for match in matched:
            contract_file = match.get('contract', {}).get('file_name', 'Unknown')
            product = match.get('contract_product', 'Unknown')
            opps = match.get('opportunities', [])
            
            print(f"\n• {contract_file} ({product})")
            for opp in opps:
                opp_num = opp.get('opportunity_number', '?')
                print(f"  → CRM #{opp_num}: {opp.get('opportunity_name', 'Unknown')}")
                print(f"    Owner: {opp.get('owner', 'Unknown')}")
                print(f"    Next Steps: {opp.get('next_steps', 'None')}")
    
    if unmatched:
        print(f"\n{'='*80}")
        print("UNMATCHED CONTRACTS (No CRM Entry)")
        print("="*80)
        for unmatch in unmatched:
            contract_file = unmatch.get('contract', {}).get('file_name', 'Unknown')
            product = unmatch.get('contract_product', 'Unknown')
            print(f"• {contract_file} ({product})")
else:
    print("\nNo matching data available or error occurred")
    if matching_data.get('error'):
        print(f"Error: {matching_data['error']}")

CONTRACT-CRM MATCHING RESULTS

Matched Contracts: 4
Unmatched Contracts: 0

MATCHED CONTRACTS

• Confluent_IBM-5.30.2023.docx (Cognos)
  → CRM #1: Confluent Cognos
    Owner: Kylie Brittz
    Next Steps: Deal signed by CPO
  → CRM #10: Confluent Cognos Renewal
    Owner: Kylie Brittz
    Next Steps: Quote for renewal being shared with team and discussing expansion

• Confluent_IBM-7.31.2024.docx (watsonx)
  → CRM #4: Confluent watsonx ESA Expansion
    Owner: Anand Das
    Next Steps: Expand to use watsonx.data for easy data access into LLM and Agents
  → CRM #8: watsonx ESA
    Owner: Kylie Brittz
    Next Steps: Meeting with their CPO to discuss larger partnership at IBM Think post initial renewal

• Confluent_IBM-1.30.2024.docx (watsonx)
  → CRM #3: Confluent watsonx ESA
    Owner: Anand Das
    Next Steps: CFO signed 1 year deal for them embedding watsonx to add agentic and AI use cases to their solution
  → CRM #6: Confluent watsonx ESA Renewal
    Owner: Anand Das
    Next Steps:

### Result Component 3B: Matching Verification & CRM Alarm Check

Run an additional matching pass to verify CRM associations, review unmatched contracts, and sound the alarm for any contract with no CRM entry.

In [9]:
# Additional verification pass: re-run Matching Agent and explicitly alarm on contracts with no CRM entry
print("\n" + "="*80)
print("MATCHING VERIFICATION & CRM ALARM CHECK")
print("="*80)

verified_matching = {}

try:
    # Reuse the same portfolio and CRM opportunity inputs shown earlier in the notebook
    verification_portfolio = contract_summary
    verification_opps = partner_profile.get("internal_data", {}).get("sales_history", {}).get("opportunities", [])

    from matching_agent import MatchingAgent
    verifier = MatchingAgent()
    verified_matching = verifier.run(verification_portfolio, verification_opps)

    if verified_matching and not verified_matching.get('error'):
        verified_matched = verified_matching.get('matched_contracts', [])
        verified_unmatched = verified_matching.get('unmatched_contracts', [])

        print(f"Verified matched contracts: {len(verified_matched)}")
        print(f"Verified unmatched contracts: {len(verified_unmatched)}")

        print("\n" + "-"*80)
        print("VERIFIED MATCHES")
        print("-"*80)
        for item in verified_matched:
            contract_file = item.get('contract_file', item.get('contract', {}).get('file_name', 'Unknown'))
            print(f"• {contract_file}")
            print(f"  Product: {item.get('contract_product', 'Unknown')}")
            print(f"  Confidence: {item.get('match_confidence', 'Unknown')}")
            print(f"  Primary CRM: {item.get('crm_opportunity', 'Unknown')}")
            print(f"  CRM Owner: {item.get('crm_owner', 'Unknown')}")
            print(f"  CRM Stage: {item.get('crm_stage', 'Unknown')}")
            print(f"  CRM Analysis: {item.get('crm_stage_analysis', 'Unknown')}")
            all_opps = item.get('opportunities', [])
            if all_opps:
                print("  All CRM opportunities linked:")
                for opp in all_opps:
                    print(f"    - CRM #{opp.get('opportunity_number', '?')}: {opp.get('opportunity_name', 'Unknown')}")
                    print(f"      Next Steps: {opp.get('next_steps', 'None')}")
            print()

        print("\n" + "-"*80)
        print("UNMATCHED CONTRACT VERIFICATION")
        print("-"*80)

        if verified_unmatched:
            for item in verified_unmatched:
                contract_file = item.get('contract_file', item.get('contract', {}).get('file_name', 'Unknown'))
                print(f"  ALARM: {contract_file}")
                print(f"  Product: {item.get('contract_product', 'Unknown')}")
                print(f"  Status: {item.get('contract_status', 'Unknown')}")
                print(f"  End Date: {item.get('contract_end', 'Unknown')}")
                print(f"  Renewal Urgency: {item.get('renewal_urgency', 'Unknown')}")
                print(f"  Alarm Message: {item.get('urgency_message', 'NO CRM ENTRY FOUND')}")
                print(f"  Required Action: {item.get('action_required', 'CREATE CRM OPPORTUNITY IMMEDIATELY')}")
                print(f"  Reason: {item.get('reason', 'No matching CRM opportunity found')}")
                print()
        else:
            print("✅ All contracts have at least one CRM association in the verification pass.")

    else:
        print("Verification pass returned no matching data or an error.")
        if verified_matching.get('error'):
            print(f"Error: {verified_matching['error']}")

except Exception as e:
    print(f"Error during matching verification: {e}")



MATCHING VERIFICATION & CRM ALARM CHECK

LLM Matching for Confluent_IBM-5.30.2023.docx:
  Matched: 2 opportunities
  Confidence: high
  Reasoning: The contract's product (Cognos) and amount ($1,000,026.00) match Opportunity 1 (CRM #1) with a produ...


Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2025-10-01)
Status code: 500, body: {"errors":[{"code":"downstream_request_failed","message":"Downstream vllm request with Model 'meta-llama/llama-3-3-70b-instruct failed: Post \"http://\u003chost\u003e:\u003cport\u003e/v1/completions\": dial tcp [::1]:3000: connect: connection refused","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"73fb7eaa07e9b563e6c486a4a97c9bd3","status_code":500}


LLM matching error: Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2025-10-01)
Status code: 500, body: {"errors":[{"code":"downstream_request_failed","message":"Downstream vllm request with Model 'meta-llama/llama-3-3-70b-instruct failed: Post \"http://\u003chost\u003e:\u003cport\u003e/v1/completions\": dial tcp [::1]:3000: connect: connection refused","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"73fb7eaa07e9b563e6c486a4a97c9bd3","status_code":500}


Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2025-10-01)
Status code: 500, body: {"errors":[{"code":"downstream_request_failed","message":"Downstream vllm request with Model 'meta-llama/llama-3-3-70b-instruct failed: Post \"http://\u003chost\u003e:\u003cport\u003e/v1/completions\": dial tcp [::1]:3000: connect: connection refused","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"5cd4c89cd12c3f505d2db0a9dfad8ba2","status_code":500}


LLM matching error: Failure during generate. (POST https://us-south.ml.cloud.ibm.com/ml/v1/text/generation?version=2025-10-01)
Status code: 500, body: {"errors":[{"code":"downstream_request_failed","message":"Downstream vllm request with Model 'meta-llama/llama-3-3-70b-instruct failed: Post \"http://\u003chost\u003e:\u003cport\u003e/v1/completions\": dial tcp [::1]:3000: connect: connection refused","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"5cd4c89cd12c3f505d2db0a9dfad8ba2","status_code":500}

LLM Matching for Confluent_IBM-1.30.2025.docx:
  Matched: 2 opportunities
  Confidence: high
  Reasoning: The contract product "watsonx" matches the CRM opportunities' products "watsonx Orchestrate, watsonx...
  Matching complete: 4 matched, 0 unmatched
Verified matched contracts: 4
Verified unmatched contracts: 0

--------------------------------------------------------------------------------
VERIFIED MATCHES
----------------------------------------------

### Result Component 4: Action Recommendations & Reasoning

View the recommended next steps and risk assessment from the Action Agent.

In [10]:
action_recommendation = my_result.get('action_recommendation', {})

print("="*80)
print("ACTION RECOMMENDATIONS & REASONING")
print("="*80)

if action_recommendation:
    # Risk Assessment
    risk_assessment = action_recommendation.get('risk_assessment', {})
    if risk_assessment:
        print("\nRISK ASSESSMENT:")
        print(f"  Risk Level: {risk_assessment.get('risk_level', 'Unknown')}")
        print(f"  Risk Score: {risk_assessment.get('risk_score', 0)}/100")
        risk_factors = risk_assessment.get('risk_factors', [])
        if risk_factors:
            print("  Risk Factors:")
            for factor in risk_factors:
                print(f"    - {factor}")
    
    # Recommended Action
    recommended_action = action_recommendation.get('recommended_action', {})
    if recommended_action:
        print(f"\n{'='*80}")
        print("RECOMMENDED NEXT STEP")
        print("="*80)
        print(f"\n{recommended_action.get('raw_recommendation', 'No recommendation available')}")
    
    # Reasoning
    reasoning = action_recommendation.get('reasoning', '')
    if reasoning:
        print(f"\n{'='*80}")
        print("REASONING")
        print("="*80)
        print(f"\n{reasoning}")
else:
    print("\nNo action recommendations available")

ACTION RECOMMENDATIONS & REASONING

RISK ASSESSMENT:
  Risk Level: High
  Risk Score: 80/100
  Risk Factors:
    - Historical deal blockers: 3 identified
    - 2 contract(s) are within the renewal window
    - 2 contract(s) expired recently

RECOMMENDED NEXT STEP

DETAILED CONTRACT ANALYSIS WITH REASONING

PRIORITY 1: CRITICAL URGENCY - IMMEDIATE ACTION REQUIRED

CONTRACT: Confluent_IBM-1.30.2025.docx
- Product(s): watsonx
- Value: $250,003.20
- Previous Signers: Unknown
- Status: EXPIRED (81 days ago)
- End Date: 2026-01-31

CRM LINKAGE:
- CRM Status: CONTRACT SIGNED
- Opportunity: "Confluent watsonx ESA"
- Owner: Anand Das
- Stage: Won
- Amount:  $250,000.00 
- Close Date: 1/31/25
- Next Steps: "CFO signed 1 year renewals"

WHY DID THIS CONTRACT EXPIRE?
- Primary Reason: Natural expiration after successful engagement
  Deal was won but contract may have expired naturally

EXPANSION OPPORTUNITY ANALYSIS:
- Can Expand: NO
- Recommendation: Focus on renewal at current scope

URGENCY ANA

## Next Steps Quality Evaluation (Governance)

Evaluate the quality and faithfulness of the recommended next steps using watsonx.governance.

In [11]:
import pandas as pd
import json

print("\n" + "="*80)
print("EVALUATING NEXT STEPS QUALITY WITH WATSONX.GOVERNANCE")
print("="*80)

# Extract next steps data from my_result
action_rec = my_result.get("action_recommendation", {})
recommended_action = action_rec.get("recommended_action", {})

# Get the ranked next steps (list of specific actions)
ranked_next_steps = recommended_action.get("ranked_next_steps", [])
# Use the first (highest priority) next step for evaluation
high_urgency_step = ranked_next_steps[0] if ranked_next_steps else recommended_action.get("raw_recommendation", "")

# Create next_steps_df for compatibility with evaluation code
all_next_steps = " | ".join(ranked_next_steps[:5]) if ranked_next_steps else high_urgency_step

next_steps_results = [{
    "message_id": "single_result",
    "input_text": my_result.get("seller_query", ""),
    "high_urgency_step": high_urgency_step,
    "all_next_steps": all_next_steps,
    "context": ""
}]

# Extract context from contract summary and partner profile
contract_summary = my_result.get("contract_summary", {})
partner_profile = my_result.get("partner_profile", {})

context_parts = []
if contract_summary:
    context_parts.append(f"Contract Summary: {json.dumps(contract_summary, default=str)}")
if partner_profile:
    context_parts.append(f"Partner Profile: {json.dumps(partner_profile, default=str)}")

context_text = " ".join(context_parts)
next_steps_results[0]["context"] = context_text

next_steps_df = pd.DataFrame(next_steps_results)

# Check if we have valid data
if ranked_next_steps and len(ranked_next_steps) > 0:
    valid_next_steps = True
else:
    valid_next_steps = False
    print("[WARNING] No ranked_next_steps found in action_recommendation")
    print(f"[DEBUG] recommended_action keys: {list(recommended_action.keys())}")

if valid_next_steps:
    # Prepare evaluation data for high urgency steps
    eval_urgency_data = pd.DataFrame({
        "input_text": [my_result.get("seller_query", "")],
        "context": [context_text],
        "generated_text": [high_urgency_step]
    })
    
    # Create faithfulness metric for next steps
    config_urgency = GenAIConfiguration(
        input_fields=["input_text"],
        context_fields=["context"],
        output_fields=["generated_text"]
    )
    
    faithfulness_urgency = FaithfulnessMetric(
        llm_judge=llm_judge,
        configuration=config_urgency
    )
    
    try:
        print(f"\nEvaluating {len(eval_urgency_data)} high urgency next steps...")
        print(f"\n[DEBUG] Next steps data shape: {eval_urgency_data.shape}")
        print(f"[DEBUG] Columns: {eval_urgency_data.columns.tolist()}")
        print(f"[DEBUG] Sample input_text: {eval_urgency_data['input_text'].iloc[0][:100]}...")
        print(f"[DEBUG] Sample context length: {len(str(eval_urgency_data['context'].iloc[0]))}")
        print(f"[DEBUG] Sample generated_text (next step): {eval_urgency_data['generated_text'].iloc[0][:100]}...")
        print(f"[DEBUG] Sample generated_text length: {len(str(eval_urgency_data['generated_text'].iloc[0]))}")
        
        eval_urgency_result = evaluator.evaluate(
            data=eval_urgency_data,
            metrics=[faithfulness_urgency]
        )
        
        print(f"\n[DEBUG] Next steps evaluation result type: {type(eval_urgency_result)}")
        print(f"[DEBUG] Has to_df: {hasattr(eval_urgency_result, 'to_df')}")
        
        if eval_urgency_result and hasattr(eval_urgency_result, 'to_df'):
            urgency_metrics_df = eval_urgency_result.to_df()
            
            print(f"[DEBUG] Next steps metrics DataFrame shape: {urgency_metrics_df.shape}")
            print(f"[DEBUG] Next steps metrics DataFrame columns: {urgency_metrics_df.columns.tolist()}")
            print(f"[DEBUG] Next steps metrics DataFrame dtypes:\n{urgency_metrics_df.dtypes}")
            print(f"[DEBUG] Next steps metrics DataFrame head:\n{urgency_metrics_df.head()}")
            
            if not urgency_metrics_df.empty:
                print(f"\n[SUCCESS] Next steps evaluation complete")
                
                # Add message_ids to metrics
                # No message_id needed for single result
                
                # Merge with next steps results
                # Store metrics for single result
                next_steps_with_metrics = urgency_metrics_df
                
                print("\n" + "="*80)
                print("NEXT STEPS EVALUATION SUMMARY")
                print("="*80)
                
                # Find metric score column
                score_candidates = [
                    'faithfulness.llm_as_judge',
                    'value',
                    'score',
                    'metric_value'
                ]
                score_col = next((col for col in score_candidates if col in urgency_metrics_df.columns), None)

                if score_col is None:
                    numeric_metric_cols = [
                        col for col in urgency_metrics_df.columns
                        if col != "message_id" and pd.api.types.is_numeric_dtype(urgency_metrics_df[col])
                    ]
                    score_col = numeric_metric_cols[0] if numeric_metric_cols else None
                
                if score_col:
                    # For single result, just use the score directly
                    avg_score = urgency_metrics_df[score_col].mean()
                    min_score = urgency_metrics_df[score_col].min()
                    max_score = urgency_metrics_df[score_col].max()
                    
                    print(f"Detected score column: {score_col}")
                    print(f"Average Next Steps Faithfulness Score: {avg_score:.2f}")
                    print(f"Min Score: {min_score:.2f}")
                    print(f"Max Score: {max_score:.2f}")
                    
                    if avg_score >= 0.8:
                        print("\nNext Steps Quality Assessment: EXCELLENT")
                    elif avg_score >= 0.6:
                        print("\nNext Steps Quality Assessment: GOOD")
                    elif avg_score >= 0.4:
                        print("\nNext Steps Quality Assessment: FAIR")
                    else:
                        print("\nNext Steps Quality Assessment: NEEDS IMPROVEMENT")
                else:
                    print("[WARNING] Could not find numeric score column in next steps evaluation")
                    next_steps_with_metrics = next_steps_df
            else:
                print("[WARNING] No metrics returned from next steps evaluation")
                next_steps_with_metrics = next_steps_df
        else:
            print("[WARNING] Next steps evaluation returned no result")
            next_steps_with_metrics = next_steps_df
            
    except Exception as e:
        print(f"[ERROR] During next steps evaluation: {str(e)}")
        import traceback
        traceback.print_exc()
        next_steps_with_metrics = next_steps_df
else:
    print("[WARNING] No valid next steps to evaluate")
    next_steps_with_metrics = next_steps_df


EVALUATING NEXT STEPS QUALITY WITH WATSONX.GOVERNANCE

Evaluating 1 high urgency next steps...

[DEBUG] Next steps data shape: (1, 3)
[DEBUG] Columns: ['input_text', 'context', 'generated_text']
[DEBUG] Sample input_text: I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what co...
[DEBUG] Sample context length: 54145
[DEBUG] Sample generated_text (next step): 1. Contact Anand Das to confirm renewal status...
[DEBUG] Sample generated_text length: 46
[Warning] No region provided : Using default region as us-south

[DEBUG] Next steps evaluation result type: <class 'ibm_watsonx_gov.entities.evaluation_result.MetricsEvaluationResult'>
[DEBUG] Has to_df: True
[DEBUG] Next steps metrics DataFrame shape: (1, 1)
[DEBUG] Next steps metrics DataFrame columns: ['faithfulness.llm_as_judge']
[DEBUG] Next steps metrics DataFrame dtypes:
faithfulness.llm_as_judge    float64
dtype: object
[DEBUG] Next steps metrics DataFrame head:
   faithfulness.llm_as_judg

Status code: 500, body: {"errors":[{"code":"downstream_request_failed","message":"Downstream vllm request with Model 'meta-llama/llama-3-3-70b-instruct failed: Post \"http://\u003chost\u003e:\u003cport\u003e/v1/completions\": dial tcp [::1]:3000: connect: connection refused","more_info":"https://cloud.ibm.com/apidocs/watsonx-ai#text-generation"}],"trace":"b852f8596cecb636e54eeae500473d49","status_code":500}


### Result Component 5: Draft Email

View the draft follow-up email generated by the Action Agent.

In [12]:
action_recommendation = my_result.get('action_recommendation', {})
draft_email = action_recommendation.get('draft_email', '')

print("="*80)
print("DRAFT FOLLOW-UP EMAIL")
print("="*80)

if draft_email:
    print(f"\n{draft_email}")
else:
    print("\nNo draft email generated")

DRAFT FOLLOW-UP EMAIL

Subject: Following up on watsonx renewal

Hi Rohan,

It was great connecting and I wanted to check in on the status of the watsonx renewal you signed off on. I'm looking forward to hearing about the progress.

Our records show the contract has expired, so I'd like to help get it back on track quickly to avoid any delays. Would it be helpful to schedule a call to discuss and answer any questions you may have? I'll be in touch with Anand to ensure we're aligned.

Regards,
[Your Name]


## Email Quality Evaluation (Governance)

Evaluate the quality and faithfulness of the generated email using watsonx.governance.

In [16]:
import pandas as pd
import json

print("="*80)
print("EVALUATING EMAIL QUALITY WITH WATSONX.GOVERNANCE")
print("="*80)

# Extract email data from my_result
action_rec = my_result.get("action_recommendation", {})
draft_email = action_rec.get("draft_email", "")
risk_assessment = action_rec.get("risk_assessment", {})
risk_level = risk_assessment.get("risk_level", "")

# Create agent_df for compatibility with evaluation code
contract_summary = my_result.get("contract_summary", {})
partner_profile = my_result.get("partner_profile", {})

# Create context from contract summary and partner profile
context_parts = []
if contract_summary:
    context_parts.append(f"Contract Summary: {json.dumps(contract_summary, default=str)}")
if partner_profile:
    context_parts.append(f"Partner Profile: {json.dumps(partner_profile, default=str)}")

agent_results = [{
    "message_id": "single_result",
    "input_text": my_result.get("seller_query", ""),
    "generated_text": draft_email,
    "risk_level": risk_level,
    "context": " ".join(context_parts)
}]

agent_df = pd.DataFrame(agent_results)

# Check if we have valid data
if draft_email and risk_level != "Error":
    valid_results = True
else:
    valid_results = False

if valid_results:
    # Prepare evaluation data
    eval_data = pd.DataFrame({
        "input_text": [my_result.get("seller_query", "")],
        "context": [" ".join(context_parts)],
        "generated_text": [draft_email]
    })
    
    # Create faithfulness metric with configuration
    config = GenAIConfiguration(
        input_fields=["input_text"],
        context_fields=["context"],
        output_fields=["generated_text"]
    )
    
    faithfulness_metric = FaithfulnessMetric(
        llm_judge=llm_judge,
        configuration=config
    )
    
    try:
        print(f"\nEvaluating {len(eval_data)} emails...")
        print(f"\n[DEBUG] Evaluation data shape: {eval_data.shape}")
        print(f"[DEBUG] Columns: {eval_data.columns.tolist()}")
        print(f"[DEBUG] Sample input_text: {eval_data['input_text'].iloc[0][:100]}...")
        print(f"[DEBUG] Sample context length: {len(str(eval_data['context'].iloc[0]))}")
        print(f"[DEBUG] Sample generated_text length: {len(str(eval_data['generated_text'].iloc[0]))}")
        
        eval_result = evaluator.evaluate(
            data=eval_data,
            metrics=[faithfulness_metric]
        )
        
        print(f"\n[DEBUG] Evaluation result type: {type(eval_result)}")
        print(f"[DEBUG] Has to_df: {hasattr(eval_result, 'to_df')}")
        
        if eval_result and hasattr(eval_result, 'to_df'):
            metrics_df = eval_result.to_df()
            
            print(f"[DEBUG] Metrics DataFrame shape: {metrics_df.shape}")
            print(f"[DEBUG] Metrics DataFrame columns: {metrics_df.columns.tolist()}")
            print(f"[DEBUG] Metrics DataFrame dtypes:\n{metrics_df.dtypes}")
            print(f"[DEBUG] Metrics DataFrame head:\n{metrics_df.head()}")
            
            if not metrics_df.empty:
                print(f"\n[SUCCESS] Evaluation complete")
                print(f"\nMetrics DataFrame:")
                print(metrics_df)
                
                # Add message_ids to metrics
                # No message_id needed for single result
                
                # Merge with agent results
                # Store metrics for single result
                final_df = metrics_df
                
                print("\n" + "="*80)
                print("EVALUATION SUMMARY")
                print("="*80)
                
                # Find metric score column
                score_candidates = [
                    'faithfulness.llm_as_judge',
                    'value',
                    'score',
                    'metric_value'
                ]
                score_col = next((col for col in score_candidates if col in metrics_df.columns), None)

                if score_col is None:
                    numeric_metric_cols = [
                        col for col in metrics_df.columns
                        if col != "message_id" and pd.api.types.is_numeric_dtype(metrics_df[col])
                    ]
                    score_col = numeric_metric_cols[0] if numeric_metric_cols else None
                
                if score_col:
                    # For single result, just use the score directly
                    avg_score = metrics_df[score_col].mean()
                    min_score = metrics_df[score_col].min()
                    max_score = metrics_df[score_col].max()
                    
                    print(f"Detected score column: {score_col}")
                    print(f"Average Faithfulness Score: {avg_score:.2f}")
                    print(f"Min Score: {min_score:.2f}")
                    print(f"Max Score: {max_score:.2f}")
                    
                    if avg_score >= 0.8:
                        print("\nOverall Assessment: EXCELLENT")
                    elif avg_score >= 0.6:
                        print("\nOverall Assessment: GOOD")
                    elif avg_score >= 0.4:
                        print("\nOverall Assessment: FAIR")
                    else:
                        print("\nOverall Assessment: NEEDS IMPROVEMENT")
                else:
                    print("[WARNING] Could not find numeric score column in results")
                    final_df = agent_df
            else:
                print("[WARNING] No metrics returned from evaluation")
                final_df = agent_df
        else:
            print("[WARNING] Evaluation returned no result")
            final_df = agent_df
            
    except Exception as e:
        print(f"[ERROR] During evaluation: {str(e)}")
        import traceback
        traceback.print_exc()
        final_df = agent_df
else:
    print("[WARNING] No valid results to evaluate")
    final_df = agent_df

EVALUATING EMAIL QUALITY WITH WATSONX.GOVERNANCE

Evaluating 1 emails...

[DEBUG] Evaluation data shape: (1, 3)
[DEBUG] Columns: ['input_text', 'context', 'generated_text']
[DEBUG] Sample input_text: I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what co...
[DEBUG] Sample context length: 54145
[DEBUG] Sample generated_text length: 486

[DEBUG] Evaluation result type: <class 'ibm_watsonx_gov.entities.evaluation_result.MetricsEvaluationResult'>
[DEBUG] Has to_df: True
[DEBUG] Metrics DataFrame shape: (1, 1)
[DEBUG] Metrics DataFrame columns: ['faithfulness.llm_as_judge']
[DEBUG] Metrics DataFrame dtypes:
faithfulness.llm_as_judge    float64
dtype: object
[DEBUG] Metrics DataFrame head:
   faithfulness.llm_as_judge
0                     0.6667

[SUCCESS] Evaluation complete

Metrics DataFrame:
   faithfulness.llm_as_judge
0                     0.6667

EVALUATION SUMMARY
Detected score column: faithfulness.llm_as_judge
Average Faithfulness Scor

### Result Component 6: CRM Updates

View the proposed CRM updates from the Action Agent.

In [14]:
import json

action_recommendation = my_result.get('action_recommendation', {})
crm_updates = action_recommendation.get('crm_updates', {})

print("="*80)
print("CRM UPDATES (Demo)")
print("="*80)

if crm_updates:
    print(f"\n{json.dumps(crm_updates, indent=2)}")
else:
    print("\nNo CRM updates generated")

CRM UPDATES (Demo)

{
  "opportunity_name": "Confluent - Seller Inquiry 2026-04-22",
  "stage": "Seller Inquiry",
  "next_step": "1. Contact Anand Das to confirm renewal status",
  "owner": "IBM Seller",
  "due_date": "2026-04-29",
  "priority": "High",
  "contracts_reviewed": 4,
  "seller_query": "I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps",
  "potential_products": "Confluent IBM-5.30.2023, Confluent IBM-7.31.2024, Cognos, watsonx.governance, watsonx Orchestrate",
  "updated_timestamp": "2026-04-22T15:33:52.044104",
  "crm_file_updated": false,
  "crm_update_disabled": true
}


### Step 2: Review and Edit the Draft Email

The initial workflow generated a draft email. You can now request edits or refinements to that email.

In [15]:
initial_email = my_result.get("action_recommendation", {}).get("draft_email", "No email generated")

print("="*80)
print("ORIGINAL DRAFT EMAIL")
print("="*80)
print(f"\n{initial_email}\n")

followup_query = "Can you make the email more urgent and add a specific deadline of April 15th for the response?"

print("="*80)
print("EMAIL EDIT REQUEST")
print("="*80)
print(f"\n{followup_query}\n")
print("="*80)
print("Generating edited email...")
print("="*80)

from langchain_ibm import WatsonxLLM
from langchain_core.prompts import ChatPromptTemplate
import re

llm = WatsonxLLM(
    model_id="meta-llama/llama-4-maverick-17b-128e-instruct-fp8",
    url="https://us-south.ml.cloud.ibm.com",
    apikey=os.getenv("WATSONX_APIKEY"),
    project_id=os.getenv("WATSONX_PROJECT_ID"),
    params={
        "max_new_tokens": 600,
        "temperature": 0.3,
        "decoding_method": "sample",
        "stop_sequences": ["```", "\n\n\n\n"]
    }
)

edit_prompt = ChatPromptTemplate.from_template(
    "You are a professional email editor. Edit the following email based on the user's request.\n\n"
    "ORIGINAL EMAIL:\n{original_email}\n\n"
    "USER REQUEST: {edit_request}\n\n"
    "INSTRUCTIONS:\n"
    "- Generate ONLY ONE complete email (no multiple versions or alternatives)\n"
    "- Start directly with 'Subject:' - no preamble or introduction\n"
    "- Do NOT repeat phrases or content\n"
    "- End cleanly after the signature line 'IBM Seller'\n"
    "- Do NOT add explanations, meta-commentary, or markdown formatting\n"
    "- Maintain professional business tone\n\n"
    "EDITED EMAIL:"
)

formatted_prompt = edit_prompt.invoke({
    "original_email": initial_email,
    "edit_request": followup_query
})

edited_email = llm.invoke(formatted_prompt)
edited_email_text = edited_email.content if hasattr(edited_email, "content") else str(edited_email)

def clean_email_output(email_text):
    email_text = email_text.strip()
    email_text = re.sub(r'^```.*?\n', '', email_text, flags=re.MULTILINE)
    email_text = re.sub(r'```.*?$', '', email_text, flags=re.MULTILINE)
    
    subject_match = re.search(r'^Subject:', email_text, flags=re.MULTILINE | re.IGNORECASE)
    if subject_match:
        email_text = email_text[subject_match.start():]
    
    signature_pattern = r'(IBM Seller)'
    match = re.search(signature_pattern, email_text)
    if match:
        email_text = email_text[:match.end()]
    
    lines = email_text.split('\n')
    cleaned_lines = []
    prev_line = None
    for line in lines:
        if line.strip() != prev_line:
            cleaned_lines.append(line)
            prev_line = line.strip()
    
    email_text = '\n'.join(cleaned_lines)
    email_text = re.sub(r'\n{3,}', '\n\n', email_text)
    
    return email_text.strip()

edited_email_text = clean_email_output(edited_email_text)

print("\n" + "="*80)
print("EDITED EMAIL")
print("="*80 + "\n")
print(edited_email_text)

ORIGINAL DRAFT EMAIL

Subject: Following up on watsonx renewal

Hi Rohan,

It was great connecting and I wanted to check in on the status of the watsonx renewal you signed off on. I'm looking forward to hearing about the progress.

Our records show the contract has expired, so I'd like to help get it back on track quickly to avoid any delays. Would it be helpful to schedule a call to discuss and answer any questions you may have? I'll be in touch with Anand to ensure we're aligned.

Regards,
[Your Name]

EMAIL EDIT REQUEST

Can you make the email more urgent and add a specific deadline of April 15th for the response?

Generating edited email...

EDITED EMAIL

Subject: Urgent: watsonx Renewal Update Required by April 15th

Hi Rohan,

I'm following up on the watsonx renewal you previously signed off on, as our records indicate the contract has expired. To avoid any potential disruptions, it's crucial we move forward promptly.

I'd appreciate it if you could confirm the status and let me 